# Self-attention, step by step[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astudentfromsustech/agents-course/blob/main/practice/p0_attention/walkthrough.ipynb)*Runs in the browser with nothing installed — or locally with `jupyter lab`.*Run each cell, look at the numbers, then **change them and run again**. That is thepoint of this notebook — the shapes are deliberately tiny (4 tokens, `d_model=6`,2 heads) so every matrix prints in full and you can check the arithmetic by hand.Three cells ask you to verify a number yourself before the notebook confirms it.Do at least one. It is the difference between reading and knowing.Companion to chapter 01 of the course. Needs only `numpy`.

In [ ]:
import numpy as npnp.set_printoptions(precision=2, suppress=True, linewidth=120)TOK = ["The", "cat", "sat", "down"]n, d_model, H = 4, 6, 2d_head = d_model // Hdef show(name, M, note=""):    print(f"{name}   shape {M.shape}" + (f"   {note}" if note else ""))    for i, row in enumerate(np.atleast_2d(M)):        label = f"{TOK[i]:>5} |" if M.shape[0] == n else "      |"        print(f"  {label} {row}")

## 1 · The inputFour tokens, already vectors — embedding happened before attention. Attention neversees words; it sees these rows and asks *which rows should influence which*.

In [ ]:
rng = np.random.default_rng(1)X = np.round(rng.normal(size=(n, d_model)), 1)show("X  (input)", X, "one row per token")

## 2 · The three weight matricesThe only parameters in attention. The **same** matrices are applied to every token —attention has no per-position weights.> **Try:** change the `0.5` scale below and watch how much it changes the attention> pattern in step 7. Large weights → sharper attention.

In [ ]:
rng = np.random.default_rng(2)Wq, Wk, Wv = (np.round(rng.normal(size=(d_model, d_model)) * 0.5, 1) for _ in range(3))show("W_q", Wq)

## 3 · Project X into Q, K, VEach token produces three vectors **from its own row only**:| | ||---|---|| **query** | what am I looking for? || **key** | what do I advertise? || **value** | what do I pass on if chosen? |

In [ ]:
Q, K, V = X @ Wq, X @ Wk, X @ Wvshow("Q", Q); print(); show("K", K); print(); show("V", V)# ✋ CHECK: row 'cat' of Q should equal X[1] @ Wqhand = X[1] @ Wqprint(f"\nby hand -> {np.round(hand,2)}")print(f"from Q   -> {np.round(Q[1],2)}   match: {np.allclose(hand, Q[1])}")

## 4 · Scores — every query against every key`scores[i,j]` is how much token *i* wants token *j*.**This is the only place information moves between tokens.** Everything else in atransformer processes each position alone.

In [ ]:
scores = Q @ K.Tshow("scores = Q @ K.T", scores, "rows = queries, cols = keys")# ✋ CHECK: scores[1,0] is the dot product of query 'cat' with key 'The'hand = float(np.dot(Q[1], K[0]))print(f"\nby hand -> {hand:.2f}    from matrix -> {scores[1,0]:.2f}   match: {np.isclose(hand, scores[1,0])}")

## 5 · Scale by √d_kHarmless at this size. At `d_k = 256` the unscaled version saturates softmax into ahard argmax — `attention_lab.py` step 2 measures that collapse directly.> **Try:** set `scale = 1.0` (no scaling) and look at how much sharper step 7 becomes.

In [ ]:
scale = np.sqrt(d_model)scaled = scores / scaleprint(f"max|score|  before {np.abs(scores).max():.2f}   after {np.abs(scaled).max():.2f}")show("scaled", scaled)

## 6 · Mask the futureToken *i* may not look at anything after *i*. `-inf` becomes exactly 0 after softmax.> **Try:** comment out the mask and watch row `The` in step 7 stop being `[1,0,0,0]`.

In [ ]:
mask = np.tril(np.ones((n, n), dtype=bool))masked = np.where(mask, scaled, -np.inf)show("mask (1 = allowed)", mask.astype(int)); print()show("scores after masking", masked)

## 7 · Softmax → attention weightsRaw scores become weights that sum to 1. **Each row is "which words are relevant to me".**

In [ ]:
e = np.exp(masked - masked.max(-1, keepdims=True))A = e / e.sum(-1, keepdims=True)show("A  (attention weights)", A)print(f"\nrow sums -> {A.sum(1)}   all 1.0: {np.allclose(A.sum(1), 1)}")print(f"\n'sat' draws from The/cat/sat as {np.round(A[2][:3],2)}")

## 8 · Output — the weighted sumEach output row is a weighted average of the **value** rows, using that row's weights.That is self-attention, complete.

In [ ]:
out = A @ Vshow("output = A @ V", out)# ✋ CHECK: build row 'sat' by hand as a weighted sum of Vhand = sum(A[2, j] * V[j] for j in range(n))for j in range(3):    print(f"   {A[2,j]:.2f} x V[{TOK[j]}]")print(f"\nby hand  -> {np.round(hand,2)}")print(f"from A@V -> {np.round(out[2],2)}   match: {np.allclose(hand, out[2])}")

## 9 · Multi-head — split, attend separatelySame Q, K, V — now cut into 2 slices of width 3, each attended **independently**.Head 0's queries never touch head 1's keys. That separation is the whole of multi-head,and it is what `attention_lab.py` tests by perturbing one head and demanding the otherscome out bit-identical.> **Try:** set `H = 3` at the top (6 ÷ 3 = 2-wide heads) and re-run from cell 1.> More heads, narrower each. What happens to how sharp each head's attention is?

In [ ]:
As, outs = [], []for h in range(H):    sl = slice(h * d_head, (h + 1) * d_head)    sc = Q[:, sl] @ K[:, sl].T / np.sqrt(d_head)    sc = np.where(mask, sc, -np.inf)    ee = np.exp(sc - sc.max(-1, keepdims=True))    a = ee / ee.sum(-1, keepdims=True)    As.append(a); outs.append(a @ V[:, sl])    show(f"A (head {h})", a); print()print("Two different opinions about relevance, from the same input.")

## 10 · Concatenate and project`W_o` is where the heads finally mix — and the **only** place they do.That is one attention layer, end to end. A transformer stacks this with a feed-forwardblock and a residual add, N times. Nothing else is new.

In [ ]:
cat = np.concatenate(outs, axis=-1)rng = np.random.default_rng(3)Wo = np.round(rng.normal(size=(d_model, d_model)) * 0.5, 1)show("concat(head0, head1)", cat, f"{d_head} + {d_head} = {d_model}")print(); show("final = concat @ W_o", cat @ Wo)

---## Now write it yourselfThis notebook showed you the mechanism. `attention_lab.py` in the same folder makes you**build** it — three `TODO`s, checked against a longhand reference loop, with behaviouraltests for the two properties that matter (heads don't mix; masking makes the pastindependent of the future).```bashpython3 attention_lab.py```Then `p1_induction_heads/` trains a tiny transformer until it discovers an inductioncircuit, and scores every head against a positional control.